
# Leaf Disease Detection — Reproducible MobileNetV2 Rebuild

This notebook runs in Google Colab or local Jupyter/VS Code. It rebuilds the leaf image classifier from the raw dataset: inspection, reproducible stratified splitting, preprocessing, Custom CNN baseline, frozen MobileNetV2, fine-tuning, validation-based selection, final evaluation, artifact saving, and new-image prediction.

The previous experiment reported 89.67% test accuracy. That is historical context only; metrics below must come from this notebook's execution and the old score is not guaranteed.

**Test-set protocol:** training and model selection use training/validation data only. The best of the three runs is selected using validation accuracy (validation macro-F1 breaks ties). The selected model is evaluated on the test data once. Therefore, test metrics for the two unselected models are intentionally blank.



## 1. Project introduction

Images are resized to 224 × 224 RGB and use MobileNetV2 preprocessing with sparse integer labels. The CNN is a baseline. MobileNetV2 first trains with its ImageNet backbone frozen, then fine-tunes selected deeper non-BatchNormalization layers at a low learning rate. Augmentation runs only when a model is training.


## 2. Google Drive mounting and configuration

In [ ]:

from pathlib import Path
import os
import json, random, time, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
from IPython.display import display
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_score, recall_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    from google.colab import drive
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
    DEFAULT_DATASET_DIR = DRIVE_ROOT / "REPLACE_WITH_DATASET_FOLDER"
    DEFAULT_ARTIFACT_DIR = DRIVE_ROOT / "leaf_disease_rebuild_artifacts"
else:
    DRIVE_ROOT = None
    DEFAULT_DATASET_DIR = Path.home() / "Desktop/DL Dataset 2/5 crop Disease and Healthy Leaf dataset"
    DEFAULT_ARTIFACT_DIR = Path.home() / "Desktop/DL Dataset 2/leaf_disease_rebuild_artifacts"

# Override these paths if your local dataset or artifact folder is elsewhere.
DATASET_DIR = Path(os.environ.get("LEAF_DATASET_DIR", str(DEFAULT_DATASET_DIR))).expanduser()
ARTIFACT_DIR = Path(os.environ.get("LEAF_ARTIFACT_DIR", str(DEFAULT_ARTIFACT_DIR))).expanduser()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_NAMES = ("train", "validation", "test")
CHECK_EXACT_DUPLICATES = True

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        warnings.warn(f"Could not enable GPU memory growth: {exc}")
print("TensorFlow:", tf.__version__)
print("GPU devices:", gpus if gpus else "None (CPU will be used)")
if DRIVE_ROOT is not None and DRIVE_ROOT.is_dir():
    print("Google Drive/MyDrive entries:")
    for entry in sorted(DRIVE_ROOT.iterdir()):
        print(("DIR " if entry.is_dir() else "FILE"), entry.name)
else:
    print("Running outside Colab; configure DATASET_DIR for this computer.")
print("\nConfigured dataset path:", DATASET_DIR)
print("Artifact path:", ARTIFACT_DIR)


## 3. Dataset inspection

In [ ]:

if not DATASET_DIR.is_dir():
    raise FileNotFoundError(
        f"Set DATASET_DIR to your Drive folder. Not found: {DATASET_DIR}"
    )
print("Dataset root entries:")
for entry in sorted(DATASET_DIR.iterdir()):
    print(("DIR " if entry.is_dir() else "FILE"), entry.name)

present = {name: (DATASET_DIR / name).is_dir() for name in SPLIT_NAMES}
if any(present.values()) and not all(present.values()):
    raise ValueError(
        f"Only some train/validation/test folders exist: {present}. "
        "Refusing to mix existing and newly generated splits."
    )
DATA_LAYOUT = "existing_splits" if all(present.values()) else "class_folders"
roots = (
    {name: DATASET_DIR / name for name in SPLIT_NAMES}
    if DATA_LAYOUT == "existing_splits" else {"raw": DATASET_DIR}
)
print("\nDetected layout:", DATA_LAYOUT)

def inspect_class_dir(class_dir):
    supported, unsupported = [], []
    for path in sorted(class_dir.rglob("*")):
        if (not path.is_file()
                or any(part.startswith(".") for part in path.relative_to(class_dir).parts)):
            continue
        if path.suffix.lower() in SUPPORTED_EXTENSIONS:
            supported.append(path.resolve())
        else:
            unsupported.append(path)
    valid, corrupt = [], []
    for path in supported:
        try:
            with Image.open(path) as image:
                image.verify()
            valid.append(path)
        except (UnidentifiedImageError, OSError, ValueError, SyntaxError):
            corrupt.append(path)
    return {"paths": valid, "unsupported": unsupported, "corrupt": corrupt}

def inspect_root(root):
    class_dirs = sorted(
        p for p in root.iterdir() if p.is_dir() and not p.name.startswith(".")
    )
    if not class_dirs:
        raise ValueError(f"No class subfolders found under {root}")
    return {folder.name: inspect_class_dir(folder) for folder in class_dirs}

if DATA_LAYOUT == "existing_splits":
    inspected = {split: inspect_root(root) for split, root in roots.items()}
    class_sets = {split: set(inspected[split]) for split in SPLIT_NAMES}
    class_names = sorted(set.union(*class_sets.values()))
    missing = {split: sorted(set(class_names) - class_sets[split]) for split in SPLIT_NAMES}
    if any(missing.values()):
        raise ValueError(f"Classes missing from existing splits: {missing}")
else:
    inspected_raw = inspect_root(DATASET_DIR)
    class_names = sorted(inspected_raw)

rows = []
if DATA_LAYOUT == "existing_splits":
    for split in SPLIT_NAMES:
        for name in class_names:
            item = inspected[split][name]
            rows.append({
                "Split": split, "Class": name, "Readable images": len(item["paths"]),
                "Unsupported files": len(item["unsupported"]),
                "Corrupt images": len(item["corrupt"]),
            })
else:
    for name in class_names:
        item = inspected_raw[name]
        rows.append({
            "Split": "raw", "Class": name, "Readable images": len(item["paths"]),
            "Unsupported files": len(item["unsupported"]),
            "Corrupt images": len(item["corrupt"]),
        })
inspection_df = pd.DataFrame(rows)
display(inspection_df)
print("Classes:", class_names)
print("Class count:", len(class_names))
print("Unsupported files:", int(inspection_df["Unsupported files"].sum()))
print("Corrupt/unreadable images:", int(inspection_df["Corrupt images"].sum()))

if DATA_LAYOUT == "existing_splits":
    empty = [
        (split, name) for split in SPLIT_NAMES for name in class_names
        if not inspected[split][name]["paths"]
    ]
    if empty:
        raise ValueError(f"Empty class folder(s) in existing splits: {empty}")
else:
    class_files = {name: inspected_raw[name]["paths"] for name in class_names}
    empty = [name for name, paths in class_files.items() if not paths]
    if empty:
        raise ValueError(f"No readable supported images in class folder(s): {empty}")


## 4. Exploratory data analysis

In [ ]:

if DATA_LAYOUT == "existing_splits":
    count_rows = [
        {"Split": split, "Class": name, "Images": len(inspected[split][name]["paths"])}
        for split in SPLIT_NAMES for name in class_names
    ]
else:
    count_rows = [
        {"Split": "raw", "Class": name, "Images": len(class_files[name])}
        for name in class_names
    ]
counts_df = pd.DataFrame(count_rows)
display(counts_df.pivot(index="Class", columns="Split", values="Images").fillna(0).astype(int))
class_counts = counts_df.groupby("Class")["Images"].sum().reindex(class_names).astype(int)
print("Total readable images:", int(class_counts.sum()))
print("Smallest/largest class:", int(class_counts.min()), "/", int(class_counts.max()))
imbalance_ratio = float(class_counts.max() / class_counts.min())
print("Largest/smallest class ratio:", round(imbalance_ratio, 2))
ax = class_counts.sort_values().plot(
    kind="barh", figsize=(10, max(5, len(class_names) * 0.35))
)
ax.set(title="Readable images per class", xlabel="Image count")
plt.tight_layout()
plt.show()


## 5. Dataset splitting and leakage checks

In [ ]:

class_to_index = {name: index for index, name in enumerate(class_names)}

def sha256_file(file_path):
    digest = hashlib.sha256()

    with open(file_path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


records = []

if DATA_LAYOUT == "existing_splits":

    # Reuse the existing train, validation, and test folders.
    # This branch does not create a new 80:10:10 split.
    for split in SPLIT_NAMES:
        for name in class_names:
            for path in inspected[split][name]["paths"]:
                records.append({
                    "split": split,
                    "class_name": name,
                    "label": class_to_index[name],
                    "path": str(path),
                })

else:

    paths_by_class_hash = {}
    hash_classes = {}
    hash_occurrences = []
    duplicate_files_removed = 0

    # ---------------------------------------------------------
    # 1. Detect exact duplicate images
    # ---------------------------------------------------------
    for name in class_names:

        paths_by_hash = {}

        for path in sorted(class_files[name]):

            digest = (
                sha256_file(path)
                if CHECK_EXACT_DUPLICATES
                else str(path)
            )

            hash_classes.setdefault(digest, set()).add(name)
            paths_by_hash.setdefault(digest, path)

            if CHECK_EXACT_DUPLICATES:
                hash_occurrences.append({
                    "sha256": digest,
                    "class_name": name,
                    "path": str(path),
                })

        duplicate_files_removed += (
            len(class_files[name]) - len(paths_by_hash)
        )

        paths_by_class_hash[name] = paths_by_hash

    # ---------------------------------------------------------
    # 2. Identify identical images assigned to different classes
    # ---------------------------------------------------------
    conflicting_hashes = {
        digest
        for digest, labels in hash_classes.items()
        if len(labels) > 1
    }

    conflicting_rows = [
        row
        for row in hash_occurrences
        if row["sha256"] in conflicting_hashes
    ]

    if conflicting_rows:

        conflict_df = pd.DataFrame(conflicting_rows)

        conflict_df.to_csv(
            ARTIFACT_DIR / "conflicting_duplicate_labels.csv",
            index=False,
        )

        print(
            "Exact image contents with conflicting class labels "
            "were excluded:",
            len(conflicting_rows),
            "files across",
            len(conflicting_hashes),
            "contents.",
        )

        display(
            conflict_df.groupby("sha256").agg(
                Classes=(
                    "class_name",
                    lambda values: sorted(set(values)),
                ),
                File_count=("path", "count"),
            ).head(20)
        )

    # ---------------------------------------------------------
    # 3. Create 80:10:10 split separately for each class
    # ---------------------------------------------------------
    for name in class_names:

        unique_paths = [
            path
            for digest, path in paths_by_class_hash[name].items()
            if digest not in conflicting_hashes
        ]

        if len(unique_paths) < 10:
            raise ValueError(
                f"Class {name!r} has only {len(unique_paths)} "
                "unique readable images. At least 10 are required "
                "for a reliable 80:10:10 split."
            )

        # First split:
        # 80% training, 20% remaining
        train_paths, remainder_paths = train_test_split(
            unique_paths,
            test_size=0.20,
            random_state=SEED,
            shuffle=True,
        )

        # Second split:
        # Divide the remaining 20% into 10% validation
        # and 10% test.
        validation_paths, test_paths = train_test_split(
            remainder_paths,
            test_size=0.50,
            random_state=SEED,
            shuffle=True,
        )

        for split, subset in (
            ("train", train_paths),
            ("validation", validation_paths),
            ("test", test_paths),
        ):

            for path in subset:

                records.append({
                    "split": split,
                    "class_name": name,
                    "label": class_to_index[name],
                    "path": str(Path(path).resolve()),
                })

    print(
        "Same-label duplicate copies excluded from raw-data splits:",
        duplicate_files_removed,
    )


# -------------------------------------------------------------
# 4. Create the dataset manifest
# -------------------------------------------------------------
manifest_df = pd.DataFrame(records)

if manifest_df.empty:
    raise ValueError("No readable images were assigned to the splits.")


# -------------------------------------------------------------
# 5. Check that image paths do not overlap between splits
# -------------------------------------------------------------
overlap = {}

for left, right in (
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
):

    overlap[(left, right)] = len(
        set(manifest_df.loc[manifest_df.split == left, "path"])
        &
        set(manifest_df.loc[manifest_df.split == right, "path"])
    )


if any(overlap.values()):
    raise RuntimeError(
        f"Image paths overlap across splits: {overlap}"
    )


# -------------------------------------------------------------
# 6. Check exact duplicate image contents across splits
# -------------------------------------------------------------
if CHECK_EXACT_DUPLICATES:

    manifest_df["sha256"] = manifest_df["path"].map(sha256_file)

    duplicate_rows = manifest_df[
        manifest_df.duplicated("sha256", keep=False)
    ]

    cross_split_duplicates = []

    for digest, group in duplicate_rows.groupby("sha256"):

        if group["split"].nunique() > 1:

            cross_split_duplicates.append({
                "sha256": digest,
                "splits": sorted(group["split"].unique()),
                "paths": group["path"].tolist(),
            })

    if cross_split_duplicates:

        display(
            pd.DataFrame(cross_split_duplicates).head(20)
        )

        if DATA_LAYOUT == "existing_splits":

            raise RuntimeError(
                "Exact duplicate image contents cross existing "
                "splits. Rebuild the splits from the raw "
                "class-folder dataset to prevent leakage."
            )

        raise RuntimeError(
            "Exact duplicate content still crosses generated "
            "splits; inspect the dataset manifest and "
            "duplicate handling."
        )


# -------------------------------------------------------------
# 7. Save the dataset split manifest
# -------------------------------------------------------------
manifest_df.to_csv(
    ARTIFACT_DIR / "dataset_split_manifest.csv",
    index=False,
)


# -------------------------------------------------------------
# 8. Display image counts by split and class
# -------------------------------------------------------------
split_counts = (
    manifest_df.groupby(["split", "class_name"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=SPLIT_NAMES, columns=class_names)
)

print("Image counts by split and class:")
display(split_counts)


# -------------------------------------------------------------
# 9. Display total images in each split
# -------------------------------------------------------------
split_totals = (
    manifest_df.groupby("split")
    .size()
    .reindex(SPLIT_NAMES)
    .rename("images")
    .to_frame()
)

display(split_totals)

print("Path overlap counts:", overlap)

print(
    "Reused existing splits:",
    DATA_LAYOUT == "existing_splits",
)


## 6. Image preprocessing

In [ ]:

def decode_and_preprocess(path, label):
    image = tf.io.decode_image(
        tf.io.read_file(path), channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE, antialias=True)
    image = preprocess_input(tf.cast(image, tf.float32))
    return image, tf.cast(label, tf.int32)

def make_dataset(split, training=False):
    subset = manifest_df[manifest_df["split"] == split]
    paths = subset["path"].to_numpy(dtype=str)
    labels = subset["label"].to_numpy(dtype=np.int32)
    if not len(paths):
        raise ValueError(f"No images in split {split!r}")
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    return (
        dataset.map(decode_and_preprocess, num_parallel_calls=AUTOTUNE)
        .batch(BATCH_SIZE).prefetch(AUTOTUNE)
    )

train_ds = make_dataset("train", training=True)
validation_ds = make_dataset("validation")
test_ds = make_dataset("test")
NUM_CLASSES = len(class_names)
print("Image size:", IMAGE_SIZE, "RGB channels: 3")
print("Batch size:", BATCH_SIZE, "Classes:", NUM_CLASSES)
for batch_images, batch_labels in train_ds.take(1):
    print("Batch shapes:", batch_images.shape, batch_labels.shape)
    print("Preprocessed range:", float(tf.reduce_min(batch_images)),
          "to", float(tf.reduce_max(batch_images)))

# These layers are embedded in each model and are active only with training=True.
def make_augmentation():
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal", seed=SEED),
        layers.RandomRotation(0.05, seed=SEED),
        layers.RandomZoom(0.10, seed=SEED),
        layers.RandomContrast(0.10, seed=SEED),
    ], name="training_augmentation")


## 7. Custom CNN baseline

In [ ]:

def build_custom_cnn():
    inputs = layers.Input(shape=(*IMAGE_SIZE, 3), name="image")
    x = make_augmentation()(inputs)
    for filters in (32, 64, 128):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.20)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="custom_cnn")

def make_callbacks(checkpoint_path, patience=4):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            str(checkpoint_path), monitor="val_accuracy", mode="max",
            save_best_only=True, verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", mode="max", patience=patience,
            restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7, verbose=1,
        ),
    ]

histories, results, checkpoint_paths = {}, {}, {}
cnn_path = ARTIFACT_DIR / "custom_cnn_best.keras"
custom_cnn = build_custom_cnn()
custom_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy", metrics=["accuracy"],
)
started = time.perf_counter()
cnn_history = custom_cnn.fit(
    train_ds, validation_data=validation_ds, epochs=15,
    callbacks=make_callbacks(cnn_path), verbose=1,
)
cnn_seconds = time.perf_counter() - started
histories["Custom CNN"] = cnn_history.history
checkpoint_paths["Custom CNN"] = cnn_path


## 8. MobileNetV2 transfer learning — frozen backbone

In [ ]:

def build_mobilenet_model():
    inputs = layers.Input(shape=(*IMAGE_SIZE, 3), name="image")
    x = make_augmentation()(inputs)
    backbone = MobileNetV2(
        weights="imagenet", include_top=False,
        input_shape=(*IMAGE_SIZE, 3), name="mobilenetv2_backbone",
    )
    backbone.trainable = False
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="mobilenetv2_leaf_classifier")

mobilenet_model = build_mobilenet_model()
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy", metrics=["accuracy"],
)
frozen_path = ARTIFACT_DIR / "mobilenetv2_frozen_best.keras"
started = time.perf_counter()
frozen_history = mobilenet_model.fit(
    train_ds, validation_data=validation_ds, epochs=12,
    callbacks=make_callbacks(frozen_path), verbose=1,
)
frozen_seconds = time.perf_counter() - started
histories["MobileNetV2 - Frozen"] = frozen_history.history
checkpoint_paths["MobileNetV2 - Frozen"] = frozen_path


## 9. MobileNetV2 fine-tuning

In [ ]:

# Continue from the best frozen-backbone checkpoint.
mobilenet_model = tf.keras.models.load_model(frozen_path)
backbone = mobilenet_model.get_layer("mobilenetv2_backbone")
backbone.trainable = True
fine_tune_layers = 30
unfreeze_from = max(0, len(backbone.layers) - fine_tune_layers)
for index, layer in enumerate(backbone.layers):
    layer.trainable = (
        index >= unfreeze_from
        and not isinstance(layer, tf.keras.layers.BatchNormalization)
    )
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy", metrics=["accuracy"],
)
fine_tuned_path = ARTIFACT_DIR / "mobilenetv2_finetuned_best.keras"
started = time.perf_counter()
fine_tuned_history = mobilenet_model.fit(
    train_ds, validation_data=validation_ds, epochs=12,
    callbacks=make_callbacks(fine_tuned_path), verbose=1,
)
fine_tuned_seconds = time.perf_counter() - started
histories["MobileNetV2 - Fine-tuned"] = fine_tuned_history.history
checkpoint_paths["MobileNetV2 - Fine-tuned"] = fine_tuned_path
print("Trainable backbone layers:", sum(layer.trainable for layer in backbone.layers))


### Training curves and validation-based model selection

In [ ]:

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["accuracy"], label="Train")
    axes[0].plot(history.history["val_accuracy"], label="Validation")
    axes[0].set(title=f"{title} accuracy", xlabel="Epoch", ylabel="Accuracy")
    axes[0].legend()
    axes[1].plot(history.history["loss"], label="Train")
    axes[1].plot(history.history["val_loss"], label="Validation")
    axes[1].set(title=f"{title} loss", xlabel="Epoch", ylabel="Loss")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(cnn_history, "Custom CNN")
plot_history(frozen_history, "MobileNetV2 frozen")
plot_history(fine_tuned_history, "MobileNetV2 fine-tuned")

def get_validation_metrics(model, history, training_seconds):
    probabilities = model.predict(validation_ds, verbose=0)
    y_true = np.concatenate([labels.numpy() for _, labels in validation_ds])
    y_pred = probabilities.argmax(axis=1)
    best_epoch = int(np.argmax(history.history["val_accuracy"]))
    return {
        "train_accuracy": float(history.history["accuracy"][best_epoch]),
        "validation_accuracy": float(history.history["val_accuracy"][best_epoch]),
        "validation_macro_f1": float(f1_score(
            y_true, y_pred, labels=np.arange(NUM_CLASSES),
            average="macro", zero_division=0,
        )),
        "training_time_seconds": float(training_seconds),
        "best_epoch": best_epoch + 1,
    }

custom_cnn = tf.keras.models.load_model(cnn_path)
results["Custom CNN"] = get_validation_metrics(custom_cnn, cnn_history, cnn_seconds)
del custom_cnn
tf.keras.backend.clear_session()
mobilenet_model = tf.keras.models.load_model(frozen_path)
results["MobileNetV2 - Frozen"] = get_validation_metrics(
    mobilenet_model, frozen_history, frozen_seconds
)
del mobilenet_model
tf.keras.backend.clear_session()
mobilenet_model = tf.keras.models.load_model(fine_tuned_path)
results["MobileNetV2 - Fine-tuned"] = get_validation_metrics(
    mobilenet_model, fine_tuned_history, fine_tuned_seconds
)
del mobilenet_model
tf.keras.backend.clear_session()

selected_model_name = max(
    results,
    key=lambda name: (
        results[name]["validation_accuracy"],
        results[name]["validation_macro_f1"],
    ),
)
selected_checkpoint = checkpoint_paths[selected_model_name]
print("Selected using validation accuracy (macro-F1 breaks ties):", selected_model_name)
display(pd.DataFrame(results).T.sort_values("validation_accuracy", ascending=False))


## 10. Final model evaluation — one test-set pass

In [ ]:

# All test metrics below are derived from this single pass over test_ds.
final_model = tf.keras.models.load_model(selected_checkpoint)
y_parts, probability_parts = [], []
loss_sum, example_count = 0.0, 0
for images, labels in test_ds:
    probabilities = final_model(images, training=False).numpy()
    labels_np = labels.numpy()
    losses = tf.keras.losses.sparse_categorical_crossentropy(
        labels_np, probabilities
    ).numpy()
    y_parts.append(labels_np)
    probability_parts.append(probabilities)
    loss_sum += float(losses.sum())
    example_count += len(labels_np)

y_test = np.concatenate(y_parts)
y_probabilities = np.concatenate(probability_parts)
y_pred = y_probabilities.argmax(axis=1)
test_loss = loss_sum / example_count
test_accuracy = accuracy_score(y_test, y_pred)
test_precision_macro = precision_score(
    y_test, y_pred, labels=np.arange(NUM_CLASSES), average="macro", zero_division=0
)
test_recall_macro = recall_score(
    y_test, y_pred, labels=np.arange(NUM_CLASSES), average="macro", zero_division=0
)
test_f1_macro = f1_score(
    y_test, y_pred, labels=np.arange(NUM_CLASSES), average="macro", zero_division=0
)
test_f1_weighted = f1_score(y_test, y_pred, average="weighted", zero_division=0)
cm = confusion_matrix(y_test, y_pred, labels=np.arange(NUM_CLASSES))
report_dict = classification_report(
    y_test, y_pred, labels=np.arange(NUM_CLASSES), target_names=class_names,
    output_dict=True, zero_division=0,
)
support = cm.sum(axis=1)
per_class_accuracy = np.divide(
    cm.diagonal(), support, out=np.zeros(NUM_CLASSES, dtype=float), where=support != 0
)
test_metrics = {
    "selected_model": selected_model_name,
    "test_examples": int(example_count),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "macro_precision": float(test_precision_macro),
    "macro_recall": float(test_recall_macro),
    "macro_f1": float(test_f1_macro),
    "weighted_f1": float(test_f1_weighted),
}
display(pd.DataFrame([test_metrics]))
print(classification_report(
    y_test, y_pred, labels=np.arange(NUM_CLASSES),
    target_names=class_names, zero_division=0,
))
per_class_df = pd.DataFrame({
    "Class": class_names, "Correct": cm.diagonal(),
    "Support": support, "Accuracy": per_class_accuracy,
})
display(per_class_df)


### Confusion matrix and inference timing

In [ ]:

fig, ax = plt.subplots(
    figsize=(max(10, NUM_CLASSES * 0.8), max(8, NUM_CLASSES * 0.65))
)
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_title(f"Test confusion matrix — {selected_model_name}")
fig.colorbar(im, ax=ax)
ax.set_xticks(np.arange(NUM_CLASSES), labels=class_names, rotation=90)
ax.set_yticks(np.arange(NUM_CLASSES), labels=class_names)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
plt.tight_layout()
plt.show()

# Benchmark on a validation image, not the held-out test set.
benchmark_images, _ = next(iter(validation_ds))
one_image = benchmark_images[:1]
for _ in range(5):
    final_model(one_image, training=False).numpy()
repetitions = 50
started = time.perf_counter()
for _ in range(repetitions):
    final_model(one_image, training=False).numpy()
inference_seconds_per_image = (time.perf_counter() - started) / repetitions
test_metrics["inference_seconds_per_image"] = float(inference_seconds_per_image)
print(f"Batch-1 inference: {inference_seconds_per_image * 1000:.2f} ms/image")


## 11. Model comparison

In [ ]:

comparison_rows = []
for name, metrics in results.items():
    selected = name == selected_model_name
    comparison_rows.append({
        "Model": name,
        "Train accuracy": metrics["train_accuracy"],
        "Validation accuracy": metrics["validation_accuracy"],
        "Validation macro F1": metrics["validation_macro_f1"],
        "Test accuracy": test_accuracy if selected else np.nan,
        "Test macro F1": test_f1_macro if selected else np.nan,
        "Training time (seconds)": metrics["training_time_seconds"],
        "Test evaluated": selected,
    })
comparison_df = pd.DataFrame(comparison_rows).sort_values(
    "Validation accuracy", ascending=False
)
display(comparison_df)
print(
    "Blank test metrics are intentional: only the validation-selected model is "
    "evaluated on the held-out test set."
)


## 12. Save final model and artifacts

In [ ]:

FINAL_MODEL_PATH = ARTIFACT_DIR / "leaf_disease_mobilenetv2.keras"
final_model.save(FINAL_MODEL_PATH)
with open(ARTIFACT_DIR / "class_names.json", "w", encoding="utf-8") as file:
    json.dump(class_names, file, indent=2, ensure_ascii=False)
with open(ARTIFACT_DIR / "model_config.json", "w", encoding="utf-8") as file:
    json.dump({
        "selected_model": selected_model_name,
        "selection_rule": "Best validation accuracy; validation macro-F1 breaks ties",
        "dataset_layout": DATA_LAYOUT,
        "dataset_dir": str(DATASET_DIR),
        "image_size": list(IMAGE_SIZE),
        "color_channels": "RGB",
        "preprocessing": "mobilenet_v2.preprocess_input",
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "split_ratios": {"train": 0.70, "validation": 0.15, "test": 0.15},
        "class_to_index": class_to_index,
        "test_evaluated_once": True,
        "tensorflow_version": tf.__version__,
    }, file, indent=2, ensure_ascii=False)
with open(ARTIFACT_DIR / "training_history.json", "w", encoding="utf-8") as file:
    json.dump(histories, file, indent=2)
with open(ARTIFACT_DIR / "evaluation_metrics.json", "w", encoding="utf-8") as file:
    json.dump({
        "validation_results": results,
        "final_test_metrics": test_metrics,
        "comparison": comparison_df.astype(object).where(
            pd.notna(comparison_df), None
        ).to_dict(orient="records"),
    }, file, indent=2)
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(
    ARTIFACT_DIR / "confusion_matrix.csv"
)
per_class_df.to_csv(ARTIFACT_DIR / "per_class_accuracy.csv", index=False)
comparison_df.to_csv(ARTIFACT_DIR / "model_comparison.csv", index=False)
with open(ARTIFACT_DIR / "classification_report.json", "w", encoding="utf-8") as file:
    json.dump(report_dict, file, indent=2, ensure_ascii=False)
print("Final model:", FINAL_MODEL_PATH)
print("Saved artifacts:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(path.name)


## 13. Prediction on new leaf images

In [ ]:

def predict_leaf_image(image_path, model_path=FINAL_MODEL_PATH, top_k=3):
    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")
    if not 1 <= top_k <= len(class_names):
        raise ValueError(f"top_k must be between 1 and {len(class_names)}")
    with Image.open(image_path) as source:
        rgb_image = source.convert("RGB").resize(
            IMAGE_SIZE, Image.Resampling.BILINEAR
        )
        image_array = np.asarray(rgb_image, dtype=np.float32)
    input_tensor = preprocess_input(image_array)[None, ...]
    model = tf.keras.models.load_model(model_path)
    probabilities = model(input_tensor, training=False).numpy()[0]
    indices = np.argsort(probabilities)[::-1][:top_k]
    print("Predicted class:", class_names[int(indices[0])])
    print(f"Confidence: {probabilities[indices[0]]:.2%}")
    display(pd.DataFrame({
        "Class": [class_names[int(i)] for i in indices],
        "Probability": [float(probabilities[i]) for i in indices],
    }))
    plt.figure(figsize=(4, 4))
    plt.imshow(rgb_image)
    plt.axis("off")
    plt.title(class_names[int(indices[0])])
    plt.show()
    return {
        "predicted_class": class_names[int(indices[0])],
        "confidence": float(probabilities[indices[0]]),
        "top_predictions": [
            {"class": class_names[int(i)], "probability": float(probabilities[i])}
            for i in indices
        ],
    }

# Example: predict_leaf_image("/content/drive/MyDrive/path/to/new_leaf.jpg")



## 14. Conclusion

Run all cells in order. Inspect the Drive entries, set `DATASET_DIR`, and review the class folders and image-quality summary before training. For raw class-folder data, identical copies with the same label are deduplicated before splitting; identical content found under conflicting labels is excluded and saved to `conflicting_duplicate_labels.csv`. This keeps duplicate content out of multiple splits without silently assigning a disputed label. Existing splits with duplicate content across splits still stop with an error and must be rebuilt from raw class folders. The selected checkpoint is chosen using validation results, and the test split is used only for the final report. All metrics are generated at runtime.
